In [ ]:
import os


In [2]:
%pwd

'c:\\Users\\yonas\\Desktop\\Mlops_Data_Science\\wine-quality-elastic-net\\notebooks'

In [3]:
os.chdir("../")
%pwd

'c:\\Users\\yonas\\Desktop\\Mlops_Data_Science\\wine-quality-elastic-net'

In [14]:
! pip install pandas -q

In [16]:
import pandas as pd

data=pd.read_csv("artifacts/data_ingestion/winequality-red.csv")
data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [17]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [18]:
data.isnull().sum()

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

In [19]:
data.shape

(1599, 12)

In [20]:
from typing import Dict, List

##### 🔄 Import Statement

```python
from typing import Dict, List

## ❓ Why?

- More standard in industry  
- Better compatibility (older Python versions, frameworks)  
- Common in ML pipelines & enterprise code  

In [44]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataValidationConfig:
    root_dir:Path
    STATUS_FILE:str
    unzip_data_dir:Path   


    all_schema: dict[str, str]
    target_column: str
    numerical_columns: list[str]
    categorical_columns: list[str]
    expected_column_order: list[str]
    expected_column_count: int

    validation_rules: Dict[str, Dict[str, float]]



##### 🔄 Add frozen=True (VERY IMPORTANT ⭐)

```python
@dataclass(frozen=True)
## ❓ Why?

- Makes config immutable  
- Prevents accidental changes during pipeline  
- Industry best practice in MLOps  

In [45]:
from src.constants.path import CONFIG_FILE_PATH,PARAMS_FILE_PATH,SCHEMA_FILE_PATH
from src.utils.common import read_yaml, create_directories

In [38]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_roots])

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        
        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir=config.root_dir,
            STATUS_FILE=config.STATUS_FILE,
            unzip_data_dir=config.unzip_data_dir,

            all_schema=self.schema.COLUMNS,
            
            target_column=self.schema.TARGET_COLUMN.name,
            numerical_columns=self.schema.NUMERICAL_COLUMNS,
            categorical_columns=self.schema.CATEGORICAL_COLUMNS,
            expected_column_order=self.schema.COLUMN_ORDER,
            expected_column_count=self.schema.COLUMN_COUNT,
            validation_rules=self.schema.DATA_VALIDATION_RULES
        )

        return data_validation_config



In [46]:
import os
from src.logger.logger import logger

In [47]:
class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_columns(self)-> bool:
        try:

            data = pd.read_csv(self.config.unzip_data_dir)

    # ----- Columns -----
            data_cols_list = list(data.columns)
            expected_cols = self.config.expected_column_order

            all_cols = set(data.columns)
            schema_cols = set(self.config.all_schema.keys())
            numerical_cols = set(self.config.numerical_columns)
            categorical_cols = set(self.config.categorical_columns)
     # ----- Structure checks -----
            # Schema match  this one all we can  all(col in schema_cols for col in all_cols)
            schema_match = all_cols.issubset(schema_cols)

            # Column existence (required columns)
            required_match = schema_cols.issubset(all_cols)

            # Numerical columns existence
            numerical_match = numerical_cols.issubset(all_cols)

            # Categorical columns existence
            categorical_match = categorical_cols.issubset(all_cols)

            # Column count check
            count_match = len(data_cols_list) == self.config.expected_column_count

            # Column order check
            order_match = data_cols_list == expected_cols

    # ----- Data type check -----
            actual_dtypes = set((col, str(dtype)) for col, dtype in data.dtypes.items())
            expected_dtypes = set(self.config.all_schema.items())

            dtype_check = expected_dtypes.issubset(actual_dtypes)

    # ----- ML validation -----
            target_check = self.config.target_column in data.columns
     # ----- Value validation (min / max) -----
            rules = self.config.validation_rules

            min_cols = {col for col, rule in rules.items() if "min" in rule}
            max_cols = {col for col, rule in rules.items() if "max" in rule}

            min_check = all(
                data[col].min() >= rules[col]["min"]
                for col in min_cols
            )

            max_check = all(
                data[col].max() <= rules[col]["max"]
                for col in max_cols
            )

            value_check = min_check and max_check

            validation_status = (
                schema_match and
                required_match and
                numerical_match and
                categorical_match and 
                count_match and 
                order_match and 
                dtype_check and 
                target_check and 
                value_check
            )
            
            with open(self.config.STATUS_FILE, 'w') as f:
                f.write(f"Validation status: {validation_status}")

            return validation_status
        
        except Exception as e:
            raise e

    

In [48]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    data_validation.validate_all_columns()
except Exception as e:
    raise e

[2026-03-19 18:09:00,003] [INFO] [src.logger.logger] : yaml file: config\config.yaml loaded successfully
[2026-03-19 18:09:00,005] [INFO] [src.logger.logger] : yaml file: params.yaml loaded successfully
[2026-03-19 18:09:00,010] [INFO] [src.logger.logger] : yaml file: schema.yaml loaded successfully
[2026-03-19 18:09:00,013] [INFO] [src.logger.logger] : created directory at: artifacts
[2026-03-19 18:09:00,014] [INFO] [src.logger.logger] : created directory at: artifacts/data_validation
